
# Forex RL — **Refined R‑Reward Notebook**

This is a **cleaned** and **organized** notebook with the reward expressed in **R‑multiples** (risk units).  
It keeps functionality intact while removing superseded cells and adds richer metrics, charts, and a one‑click **smoke test**.

**Sections**  
1. Install & Imports  
2. Config (edit here)  
3. Data & Features  
4. Walk‑Forward Split Utilities  
5. Environment (R‑reward) + SB3 Wrapper  
6. VecEnv Builder + Scaler  
7. Metrics (R‑metrics, daily MTM, baselines, charts)  
8. Training (R‑based selection)  
9. Walk‑Forward CV Runner  
10. **Trade Report + Extended Metrics (NEW)**  
11. **Smoke Test Walk‑Forward (NEW)**



> **How to run (top-to-bottom):**  
> 1) Install, 2) Imports, 3) Config, 4) Data & features, 5) Scaler & folds, 6) Env, 7) Env builder,  
> 8) Metrics, 9) Eval callback, 10) Training override (with progress bar),  
> 11) Trade report functions, 12) **Smoke test** (quick, produces HTML),  
> 13) **Full Walk‑Forward**, 14) **WF HTML report & ZIP**.


# Forex RL (EUR/USD) — R‑Reward, Time Exit, No Friction, Walk‑Forward CV



- **No friction**: commission = 0, slippage = 0.  
- **No manual Close**: exits only via **SL/TP** or **time‑based exit** (no trailing stops yet).  
- **Reward in R units**: reward is the **realized R‑multiple at exit**; 1R is the risk set at the stop.  
- **Extended data**: attempts to pull **2015‑01‑01 → 2025‑08‑01** daily bars for `EURUSD=X` via `yfinance`.  
- **Walk‑Forward Cross‑Validation**: folds with a summary performance chart (Agent vs Buy‑&‑Hold).  
- **Sectioned layout**: run top‑to‑bottom; choose algorithm (DQN, QR‑DQN, RecurrentPPO).

> Notes:  
> * The environment is left with **3 actions**: `0=Hold`, `1=Open Long`, `2=Open Short`. No manual Close action is implemented here.  
> * Exit geometry defaults: **SL ≈ 2.0×ATR**, **TP ≈ 2.5×ATR**, **time‑based exit** after ~30 bars (tunable).  
> * If you want to re‑introduce costs or trailing stops later, search for the parameters in the **Config** section.

In [ ]:
# @title Install dependencies (run once per session)
# If you're on Colab, enable GPU: Runtime ▶ Change runtime type ▶ GPU
# (You can comment out any line you already have installed locally)

import sys

def pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core numeric + plotting + data
!pip install numpy pandas matplotlib mplfinance yfinance
# Indicators
!pip install finta
# Gym + RL
!pip install gymnasium
!pip install stable-baselines3[extra] sb3-contrib
# TensorBoard for logs
!pip install tensorboard

In [ ]:
# @title Imports
import os, math, warnings, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from finta import TA

import gymnasium as gym
from gymnasium import spaces

import torch as th
from torch import nn

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed

from sb3_contrib import QRDQN, RecurrentPPO

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
# @title Config (edit here)
SYMBOL = "EURUSD=X"
START_DATE = "2015-01-01"
END_DATE   = "2025-08-01"

# Feature window and env parameters
WINDOW_SIZE = 64
POSITION_RISK_FRAC = 0.01

# 👉 Denser exits (more trades)
SL_ATR_MULT = 1.8
TP_ATR_MULT = 2.2
MAX_BARS_IN_TRADE = 18

# Friction OFF per request
TRANSACTION_COST = 0.0
SLIPPAGE = 0.0

# Penalties
INVALID_ACTION_PENALTY = 0.0
DD_PENALTY = 0.0
TURNOVER_PENALTY = 0.0

# Training
TOTAL_TIMESTEPS = 400_000  # a bit longer to benefit from more frequent exits
EVAL_FREQ = 120_000         # evaluate less often (less noisy)
ALGO = "DQN"               # "DQN", "QRDQN", "RPPO"

# Walk-forward CV
MIN_TRAIN_MONTHS = 24
VAL_MONTHS = 9            # 👉 longer validation → stabler eval signal
TEST_MONTHS = 6
STEP_MONTHS = 12

# Validation activity guard (reject do-nothing policies)
MIN_TRADES_VAL = 4        # require at least this many trades on the val slice

LOG_DIR = "./logs_rl"
os.makedirs(LOG_DIR, exist_ok=True)

N_ENVS = 24  # number of parallel envs for training
RPPO_N_STEPS = 128  # rollout length per env for RPPO

# --- WF-CV memory controls (NEW) ---
WF_USE_ROLLING = True          # Use a rolling train window (bounded size)
WF_ROLLING_TRAIN_MONTHS = 36   # Train window length per fold (e.g., last 36 months)

# For WF-CV parallelism: by default use same as training;
# if RAM gets tight, set this lower than N_ENVS.
WF_N_ENVS = N_ENVS             # e.g., 8 or 12 (match your Config)


# --- Reporting & UI ---
USE_PROGRESS_BAR = True  # UI only; does not affect training results
REPORT_DIR = os.path.join(LOG_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)


In [ ]:
# @title Data download & features (robust to yfinance MultiIndex)
def download_eurusd(symbol=SYMBOL, start=START_DATE, end=END_DATE):
    df = yf.download(
        symbol,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column",  # try to avoid ticker-grouped columns
    )

    # --- Flatten possible MultiIndex columns (yfinance can return them) ---
    if isinstance(df.columns, pd.MultiIndex):
        try:
            # If second level is a single ticker, drop it
            if df.columns.nlevels >= 2 and len(df.columns.get_level_values(1).unique()) == 1:
                df.columns = df.columns.get_level_values(0)
            else:
                # Otherwise, join levels (e.g., 'open_EURUSD=X')
                df.columns = [
                    "_".join([str(x) for x in col if x is not None]).lower()
                    for col in df.columns
                ]
        except Exception:
            df.columns = [
                "_".join([str(x) for x in col if x is not None]).lower()
                for col in df.columns
            ]

    # Normalize column names
    df = df.rename(columns=str.lower)

    # Remove exact duplicate column names if any
    df = df.loc[:, ~df.columns.duplicated()]

    # We want open/high/low/close/volume columns. They might be like 'open' or 'open_eurusd=x'
    required = ["open", "high", "low", "close", "volume"]
    cols = {}
    for base in required:
        if base in df.columns:
            cols[base] = base
        else:
            # find first column that starts with 'base'
            matches = [c for c in df.columns if isinstance(c, str) and c.startswith(base)]
            if matches:
                cols[base] = matches[0]

    # Final safety: ensure we found them all
    missing = [b for b in required if b not in cols]
    if missing:
        raise ValueError(f"Could not locate required columns in downloaded data: {missing}. Got columns: {list(df.columns)}")

    # Select & clean
    out = df[[cols[b] for b in required]].copy()
    out.columns = required  # rename back to canonical names

    # Enforce numeric 1-D Series
    for c in required:
        out[c] = pd.to_numeric(out[c].squeeze(), errors="coerce")

    # Basic hygiene
    out = out.dropna()
    out = out.sort_index()
    return out

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # EMA 20
    out['ema20'] = out['close'].ewm(span=20, adjust=False).mean()

    # RSI 14 (Wilder)
    delta = out['close'].diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/14, adjust=False).mean()
    roll_down = down.ewm(alpha=1/14, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-12)
    out['rsi'] = 100 - (100 / (1 + rs))

    # MACD (12,26,9)
    ema12 = out['close'].ewm(span=12, adjust=False).mean()
    ema26 = out['close'].ewm(span=26, adjust=False).mean()
    out['macd'] = ema12 - ema26
    out['macd_signal'] = out['macd'].ewm(span=9, adjust=False).mean()
    out['macd_hist'] = out['macd'] - out['macd_signal']

    # Bollinger (20, 2)
    mavg = out['close'].rolling(20).mean()
    mstd = out['close'].rolling(20).std(ddof=0)
    out['bb_middle'] = mavg
    out['bb_upper'] = mavg + 2 * mstd
    out['bb_lower'] = mavg - 2 * mstd

    # ATR (Wilder 14)
    high_low = (out['high'] - out['low']).abs()
    high_close = (out['high'] - out['close'].shift()).abs()
    low_close = (out['low'] - out['close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/14, adjust=False).mean()
    out['atr'] = pd.to_numeric(atr.squeeze(), errors='coerce')

    # Normalized ATR (force 1-D)
    out['atr_n'] = out['atr'].to_numpy() / out['close'].to_numpy()

    # Diagnostics
    out['ret'] = out['close'].pct_change()

    out = out.dropna().copy()
    return out

raw = download_eurusd()
data = compute_indicators(raw)

feature_cols = [
    'ema20','rsi','macd','macd_signal','macd_hist',
    'bb_upper','bb_middle','bb_lower',
    'atr_n','ret'
]
price_cols = ['open','high','low','close','atr']

assert all(c in data.columns for c in feature_cols + price_cols), "Missing required columns."
print(f"Data rows after indicators: {len(data):,}. Range: {data.index.min().date()} → {data.index.max().date()}")
data.head()


In [ ]:
# @title Plotting the data with indicators
import mplfinance as mpf
from datetime import datetime, timedelta

def plot_ohlc_with_indicators(df: pd.DataFrame):
    """
    Plots an OHLC chart with EMA, RSI, MACD, and Bollinger Bands.
    """
    apd = [
        mpf.make_addplot(df['ema20'], color='blue', panel=0, width=0.7, ylabel='Price'),
        mpf.make_addplot(df['bb_upper'], color='cyan', linestyle='dashed', width=0.7, panel=0),
        mpf.make_addplot(df['bb_middle'], color='gray', linestyle='dashed', width=0.7, panel=0),
        mpf.make_addplot(df['bb_lower'], color='cyan', linestyle='dashed', width=0.7, panel=0),
        mpf.make_addplot(df['rsi'], color='purple', panel=1, width=0.7, ylabel='RSI'),
        mpf.make_addplot(df['macd'], color='green', panel=2, width=0.7, ylabel='MACD'),
        mpf.make_addplot(df['macd_signal'], color='red', width=0.7, panel=2),
        mpf.make_addplot(df['macd_hist'], type='bar', color='gray', width=0.7, panel=2),
        mpf.make_addplot(df['atr_n'], color='orange', width=0.7, panel=3, ylabel='ATR (Norm)')
    ]

    mpf.plot(df, type='candle', style='yahoo',
             title="EUR/USD with Indicators",
             ylabel="Price",
             addplot=apd,
             figscale=1.5,
             volume=True,
             volume_panel=4,
             ylabel_lower='Volume')

# Plot the data with indicators for the last year
last_year_data = data[data.index >= (data.index.max() - timedelta(days=200))]
plot_ohlc_with_indicators(last_year_data)

In [ ]:
# @title Scaler & Walk-forward folds
class ZScoreScaler:
    def __init__(self):
        self.means = None
        self.stds = None

    def fit(self, df_feat: pd.DataFrame):
        self.means = df_feat.mean()
        self.stds  = df_feat.std().replace(0, 1e-8)

    def transform(self, df_feat: pd.DataFrame) -> pd.DataFrame:
        return (df_feat - self.means) / self.stds

    def fit_transform(self, df_feat: pd.DataFrame) -> pd.DataFrame:
        self.fit(df_feat)
        return self.transform(df_feat)

def build_walkforward_folds(df: pd.DataFrame,
                            min_train_months=24, val_months=6, test_months=6, step_months=6):
    idx = df.index
    start = idx.min()
    end   = idx.max()

    folds = []
    # first test start after we have min_train + val
    first_test_start = start + pd.DateOffset(months=min_train_months + val_months)

    test_start = first_test_start
    k = 1
    while True:
        train_end = test_start - pd.DateOffset(months=val_months)
        val_end   = test_start
        test_end  = test_start + pd.DateOffset(months=test_months)
        if test_end > end:
            break

        if WF_USE_ROLLING:
            # Rolling train window (bounded memory)
            train_start = train_end - pd.DateOffset(months=max(min_train_months, WF_ROLLING_TRAIN_MONTHS))
            train_start = max(train_start, start)
        else:
            # Expanding window as before
            train_start = start

        folds.append({
            "fold": k,
            "train_start": train_start, "train_end": train_end,
            "val_start": train_end,     "val_end": val_end,
            "test_start": val_end,      "test_end": test_end
        })
        k += 1
        test_start = test_start + pd.DateOffset(months=step_months)

    return folds


folds = build_walkforward_folds(data, MIN_TRAIN_MONTHS, VAL_MONTHS, TEST_MONTHS, STEP_MONTHS)
print(f"Built {len(folds)} walk-forward folds.")
for f in folds[:3]:
    print(f)

In [ ]:
# @title Custom ForexEnv (R-reward, time exit, no friction, no manual Close)
class ForexEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        market_df: pd.DataFrame,
        feat_df: pd.DataFrame,
        window_size: int = 64,
        position_frac: float = 0.01,
        sl_atr_mult: float = 2.0,
        tp_atr_mult: float = 2.5,
        max_bars_in_trade: int = 30,
        transaction_cost: float = 0.0,
        slippage: float = 0.0,
        invalid_action_penalty: float = 0.0,
        dd_penalty: float = 0.0,
        turnover_penalty: float = 0.0,
        seed: int = 42
    ):
        super().__init__()
        assert len(market_df) == len(feat_df), "Market and feature frames must align."
        self.mkt = market_df.reset_index(drop=True).copy()
        self.feat = feat_df.reset_index(drop=True).copy()
        self.window_size = int(window_size)
        self.position_frac = float(position_frac)
        self.sl_atr_mult = float(sl_atr_mult)
        self.tp_atr_mult = float(tp_atr_mult)
        self.max_bars_in_trade = int(max_bars_in_trade)
        self.transaction_cost = float(transaction_cost)
        self.slippage = float(slippage)
        self.invalid_action_penalty = float(invalid_action_penalty)
        self.dd_penalty = float(dd_penalty)
        self.turnover_penalty = float(turnover_penalty)

        self.rng = np.random.default_rng(seed)

        self.n_features = self.feat.shape[1]
        self.action_space = spaces.Discrete(3)  # 0 hold, 1 long, 2 short
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.window_size, self.n_features),
            dtype=np.float32
        )

        # trade logs
        self.entries = []
        self.trades = []

        self._reset_state()

    def _reset_state(self):
        self.t = self.window_size  # start index for first obs
        self.net_worth = 1.0
        self.peak_net_worth = 1.0
        self.position = 0  # 0 flat, +1 long, -1 short
        self.entry_price = None
        self.stop_price = None
        self.take_price = None
        self.units = 0.0
        self.bars_in_trade = 0
        self.open_trade_i = None  # index of last entries record

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_state()
        obs = self._get_obs()
        info = {}
        return obs, info

    def _get_obs(self):
        s = self.t - self.window_size
        e = self.t
        window = self.feat.iloc[s:e].values.astype(np.float32)
        return window

    def _reward_from_exit(self, exit_price, reason):
        # Realized PnL in R-multiples (risk defined at stop)
        if self.position == 0 or self.entry_price is None or self.stop_price is None:
            return 0.0

        risk_per_unit = abs(self.entry_price - self.stop_price)
        risk_value = risk_per_unit * self.units
        if risk_value <= 1e-12:
            return 0.0

        profit_value = 0.0
        if self.position == +1:
            profit_value = (exit_price - self.entry_price) * self.units
        elif self.position == -1:
            profit_value = (self.entry_price - exit_price) * self.units

        realized_R = float(profit_value / (risk_value + 1e-12))

        # Update NW on exit (costs are zero per request)
        self.net_worth += profit_value

        # Drawdown penalty on *new* drawdown growth (kept gentle/off)
        self.peak_net_worth = max(self.peak_net_worth, self.net_worth)
        dd = 0.0
        if self.peak_net_worth > 0:
            dd = max(0.0, 1.0 - self.net_worth / self.peak_net_worth)

        penalty = 0.0
        if self.dd_penalty > 0:
            # Penalize proportional to drawdown level (or growth if you track prev)
            penalty += self.dd_penalty * dd

        # turnover penalty: count exit as a change
        penalty += self.turnover_penalty

        reward = realized_R - penalty

        # log trade
        self.trades.append({
            "type": "long" if self.position == +1 else "short",
            "reason": reason,
            "entry_t": self.entries[self.open_trade_i]["entry_t"] if self.open_trade_i is not None else None,
            "exit_t": self.t,
            "entry_price": self.entry_price,
            "exit_price": exit_price,
            "stop": self.stop_price,
            "take": self.take_price,
            "units": self.units,
            "realized_R": realized_R,
            "pnl_value": profit_value,
            "net_worth": self.net_worth
        })

        # reset position
        self.position = 0
        self.entry_price = None
        self.stop_price = None
        self.take_price = None
        self.units = 0.0
        self.bars_in_trade = 0
        self.open_trade_i = None

        return float(reward)

    def step(self, action):
        done = False
        info = {}
        reward = 0.0

        # Current bar prices
        row = self.mkt.iloc[self.t]
        high = float(row["high"])
        low  = float(row["low"])
        close = float(row["close"])
        atr = float(row["atr"])
        atr_abs = max(atr, 1e-6)  # ensure non-zero

        # Handle action
        if action == 1:  # Open Long
            if self.position == 0:
                self.position = +1
                self.entry_price = close + self.slippage
                stop_dist = self.sl_atr_mult * atr_abs
                take_dist = self.tp_atr_mult * atr_abs
                self.stop_price = self.entry_price - stop_dist
                self.take_price = self.entry_price + take_dist

                # position sizing to risk ~1% of equity at stop
                risk_value = self.net_worth * self.position_frac
                per_unit_risk = max(1e-6, self.entry_price - self.stop_price)
                self.units = risk_value / per_unit_risk

                self.entries.append({
                    "type":"long", "entry_t": self.t,
                    "entry_price": self.entry_price,
                    "stop": self.stop_price, "take": self.take_price,
                    "units": self.units
                })
                self.open_trade_i = len(self.entries) - 1
                # turnover penalty only when changing position from flat
                reward -= self.turnover_penalty
            else:
                reward -= self.invalid_action_penalty

        elif action == 2:  # Open Short
            if self.position == 0:
                self.position = -1
                self.entry_price = close - self.slippage
                stop_dist = self.sl_atr_mult * atr_abs
                take_dist = self.tp_atr_mult * atr_abs
                self.stop_price = self.entry_price + stop_dist
                self.take_price = self.entry_price - take_dist

                risk_value = self.net_worth * self.position_frac
                per_unit_risk = max(1e-6, self.stop_price - self.entry_price)
                self.units = risk_value / per_unit_risk

                self.entries.append({
                    "type":"short", "entry_t": self.t,
                    "entry_price": self.entry_price,
                    "stop": self.stop_price, "take": self.take_price,
                    "units": self.units
                })
                self.open_trade_i = len(self.entries) - 1
                reward -= self.turnover_penalty
            else:
                reward -= self.invalid_action_penalty
        else:
            # Hold -> no change
            pass

        # Check for exits on this bar (SL first, then TP)
        if self.position != 0:
            self.bars_in_trade += 1
            if self.position == +1:
                if low <= self.stop_price:
                    reward += self._reward_from_exit(self.stop_price, "sl")
                elif high >= self.take_price:
                    reward += self._reward_from_exit(self.take_price, "tp")
                elif self.bars_in_trade >= self.max_bars_in_trade:
                    reward += self._reward_from_exit(close, "time")
            elif self.position == -1:
                if high >= self.stop_price:
                    reward += self._reward_from_exit(self.stop_price, "sl")
                elif low <= self.take_price:
                    reward += self._reward_from_exit(self.take_price, "tp")
                elif self.bars_in_trade >= self.max_bars_in_trade:
                    reward += self._reward_from_exit(close, "time")

        # Advance time
        self.t += 1
        if self.t >= len(self.mkt) - 1:
            # End of data: force close if still open
            if self.position != 0:
                reward += self._reward_from_exit(float(self.mkt.iloc[-1]["close"]), "eod")
            done = True

        obs = self._get_obs()
        return obs, float(reward), done, False, info

    @property
    def position_frac(self):
        return self.position_frac_

    @position_frac.setter
    def position_frac(self, v):
        self.position_frac_ = float(v)

    # Helpers for analysis
    def get_trade_log(self):
        return pd.DataFrame(self.trades)

    def get_entries_log(self):
        return pd.DataFrame(self.entries)


class ForexEnvSB3(gym.Wrapper):
    """SB3 shim: flattens (window, features) -> 1D vector for MlpPolicy."""
    def __init__(self, env: ForexEnv):
        super().__init__(env)
        w, f = env.observation_space.shape
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(w*f,), dtype=np.float32
        )
        self.action_space = env.action_space

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return obs.reshape(-1).astype(np.float32), info

    def step(self, action):
        obs, reward, done, trunc, info = self.env.step(action)
        return obs.reshape(-1).astype(np.float32), reward, done, trunc, info

In [ ]:
# @title Env builder for date slices (fit scaler on train, apply to val/test) + parallelization
def _make_one_env(df_slice, scaler, seed):
    """Factory that returns a callable building ONE env instance (for VecEnv)."""
    def _thunk():
        mkt_df = df_slice[price_cols].copy()
        feat_scaled = scaler.transform(df_slice[feature_cols].copy())
        base = ForexEnv(
            market_df=mkt_df, feat_df=feat_scaled,
            window_size=WINDOW_SIZE, position_frac=POSITION_RISK_FRAC,
            sl_atr_mult=SL_ATR_MULT, tp_atr_mult=TP_ATR_MULT, max_bars_in_trade=MAX_BARS_IN_TRADE,
            transaction_cost=TRANSACTION_COST, slippage=SLIPPAGE,
            invalid_action_penalty=INVALID_ACTION_PENALTY,
            dd_penalty=DD_PENALTY, turnover_penalty=TURNOVER_PENALTY,
            seed=seed
        )
        return ForexEnvSB3(base)
    return _thunk

def make_scaled_env(train_df, eval_df, n_envs=1):
    # Fit scaler on train only
    scaler = ZScoreScaler()
    scaler.fit(train_df[feature_cols])

    # TRAIN: vectorized
    if n_envs > 1:
        fns = [_make_one_env(train_df, scaler, RANDOM_SEED + i) for i in range(n_envs)]
        train_env = SubprocVecEnv(fns)
    else:
        train_env = DummyVecEnv([_make_one_env(train_df, scaler, RANDOM_SEED)])

    # EVAL: single env for deterministic evaluation
    eval_env = DummyVecEnv([_make_one_env(eval_df, scaler, RANDOM_SEED + 999)])

    return train_env, eval_env, scaler

In [ ]:
# @title Metrics layer: R-metrics, daily MTM, baselines, plotting

import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, json, math
from typing import Tuple, Dict, List

# --------------------------
# Utility: Sharpe & DD (daily)
# --------------------------
def _daily_sharpe(daily_returns: np.ndarray, risk_free: float = 0.0) -> float:
    if daily_returns is None or len(daily_returns) < 2:
        return np.nan
    mu = np.nanmean(daily_returns) - risk_free/252.0
    sd = np.nanstd(daily_returns, ddof=1)
    if sd <= 1e-12:
        return np.nan
    return float((mu / sd) * np.sqrt(252))

def _max_drawdown(series: np.ndarray) -> float:
    if series is None or len(series) == 0:
        return np.nan
    peak = -np.inf
    max_dd = 0.0
    for v in series:
        if v > peak:
            peak = v
        dd = (v/peak - 1.0) if peak > 0 else 0.0
        if dd < max_dd:
            max_dd = dd
    return float(max_dd)  # negative

# --------------------------
# Trade simulation (baseline & MTM)
# --------------------------
def _simulate_trades_from_signals(df: pd.DataFrame,
                                  signals: pd.Series,
                                  position_frac: float,
                                  sl_atr_mult: float,
                                  tp_atr_mult: float,
                                  max_bars_in_trade: int) -> pd.DataFrame:
    """
    signals: 0=hold, 1=open long, 2=open short (open only if flat)
    Returns DataFrame with trade log containing:
      type(long/short), entry_t, exit_t, entry_price, exit_price, stop, take,
      units, realized_R, pnl_value, reason, bars_in_trade
    Uses absolute ATR for SL/TP and fixed-risk sizing (1R approx POSITION_RISK_FRAC of equity).
    Equity is updated trade-to-trade (no compounding during the trade).
    """
    eq = 1.0  # start equity (normalized)
    trades = []
    in_pos = 0
    entry_i = None
    entry_price = None
    stop = None
    take = None
    units = 0.0
    bars = 0

    o,h,l,c,atr = df["open"].values, df["high"].values, df["low"].values, df["close"].values, df["atr"].values
    idx = df.index.to_list()

    for t in range(len(df)):
        # exits first (intraday priority: SL then TP, then time)
        if in_pos != 0:
            bars += 1
            if in_pos == +1:
                if l[t] <= stop:
                    exit_p = stop; reason = "sl"
                elif h[t] >= take:
                    exit_p = take; reason = "tp"
                elif bars >= max_bars_in_trade:
                    exit_p = c[t]; reason = "time"
                else:
                    exit_p = None
                if exit_p is not None:
                    risk_per_unit = abs(entry_price - stop)
                    risk_value = risk_per_unit * units
                    profit_value = (exit_p - entry_price) * units - 0.0  # costs off
                    realized_R = (profit_value / risk_value) if risk_value > 1e-12 else 0.0
                    eq = max(1e-12, eq + realized_R * position_frac * eq)  # R on equity fraction
                    trades.append({
                        "type":"long","entry_t":idx[entry_i],"exit_t":idx[t],
                        "entry_price":float(entry_price),"exit_price":float(exit_p),
                        "stop":float(stop),"take":float(take),"units":float(units),
                        "realized_R":float(realized_R),"pnl_value":float(profit_value),
                        "reason":reason,"bars_in_trade":bars
                    })
                    in_pos = 0; entry_i=None; bars=0
            elif in_pos == -1:
                if h[t] >= stop:
                    exit_p = stop; reason = "sl"
                elif l[t] <= take:
                    exit_p = take; reason = "tp"
                elif bars >= max_bars_in_trade:
                    exit_p = c[t]; reason = "time"
                else:
                    exit_p = None
                if exit_p is not None:
                    risk_per_unit = abs(entry_price - stop)
                    risk_value = risk_per_unit * units
                    profit_value = (entry_price - exit_p) * units - 0.0  # costs off
                    realized_R = (profit_value / risk_value) if risk_value > 1e-12 else 0.0
                    eq = max(1e-12, eq + realized_R * position_frac * eq)  # R on equity fraction
                    trades.append({
                        "type":"short","entry_t":idx[entry_i],"exit_t":idx[t],
                        "entry_price":float(entry_price),"exit_price":float(exit_p),
                        "stop":float(stop),"take":float(take),"units":float(units),
                        "realized_R":float(realized_R),"pnl_value":float(profit_value),
                        "reason":reason,"bars_in_trade":bars
                    })
                    in_pos = 0; entry_i=None; bars=0

        # open if flat & signal requests it
        if in_pos == 0 and t < len(df)-1:
            sig = int(signals.iloc[t])
            if sig == 1 or sig == 2:
                # position sizing: risk a fraction of equity (1R ~= position_frac * equity)
                # absolute ATR for stop distance
                risk_per_unit = atr[t] * max(1e-12, sl_atr_mult)
                risk_capital = position_frac * eq
                units = (risk_capital / risk_per_unit) if risk_per_unit > 1e-12 else 0.0

                entry_price = c[t]
                if sig == 1:
                    stop = entry_price - atr[t] * sl_atr_mult
                    take = entry_price + atr[t] * tp_atr_mult
                    in_pos = +1
                else:
                    stop = entry_price + atr[t] * sl_atr_mult
                    take = entry_price - atr[t] * tp_atr_mult
                    in_pos = -1
                entry_i = t; bars = 0

    # end-of-data force close
    if in_pos != 0:
        exit_p = c[-1]; reason = "eod"
        if in_pos == +1:
            profit_value = (exit_p - entry_price) * units
        else:
            profit_value = (entry_price - exit_p) * units
        risk_per_unit = abs(entry_price - stop)
        risk_value = risk_per_unit * units
        realized_R = (profit_value / risk_value) if risk_value > 1e-12 else 0.0
        trades.append({
            "type":"long" if in_pos==+1 else "short","entry_t":idx[entry_i],"exit_t":idx[-1],
            "entry_price":float(entry_price),"exit_price":float(exit_p),
            "stop":float(stop),"take":float(take),"units":float(units),
            "realized_R":float(realized_R),"pnl_value":float(profit_value),
            "reason":reason,"bars_in_trade":bars
        })

    return pd.DataFrame(trades)

def equity_R_from_trades(trades: pd.DataFrame) -> np.ndarray:
    if trades is None or len(trades) == 0:
        return np.array([1.0])
    eq = [1.0]
    for r in trades["realized_R"].values:
        eq.append(eq[-1] * (1.0 + r * POSITION_RISK_FRAC))
    return np.array(eq)

def daily_mtm_from_trades(df: pd.DataFrame, trades: pd.DataFrame) -> pd.DataFrame:
    """
    Build daily mark-to-market equity in both % and R-terms based on close prices between trade dates.
    For open trades, mark to close each day; on exit dates, use realized R.
    Returns DataFrame with columns: equity_pct (starting at 1.0), daily_ret, equity_R
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()
    dates = df.index
    equity_pct = np.ones(len(df), dtype=float)
    equity_R   = np.ones(len(df), dtype=float)
    in_pos = 0; entry_price=None; stop=None; risk_per_unit=None; entry_idx=None

    trade_iter = list(trades.to_dict("records")) if trades is not None else []
    k = 0

    for i in range(len(df)):
        d = dates[i]
        # if a trade starts at this index
        if k < len(trade_iter) and trade_iter[k]["entry_t"] == d and (i == 0 or in_pos == 0):
            entry_price = trade_iter[k]["entry_price"]
            stop = trade_iter[k]["stop"]
            risk_per_unit = abs(entry_price - stop)
            in_pos = +1 if trade_iter[k]["type"] == "long" else -1

        # mark to market using close
        if in_pos != 0 and risk_per_unit and risk_per_unit > 1e-12:
            if in_pos == +1:
                uR = (df["close"].iloc[i] - entry_price) / risk_per_unit
            else:
                uR = (entry_price - df["close"].iloc[i]) / risk_per_unit
        else:
            uR = 0.0

        equity_R[i] = (equity_R[i-1] if i>0 else 1.0) * (1.0 + uR * POSITION_RISK_FRAC)

        # realize if this day is the exit
        if k < len(trade_iter) and trade_iter[k]["exit_t"] == d:
            realized_R = trade_iter[k]["realized_R"]
            equity_R[i] = (equity_R[i-1] if i>0 else 1.0) * (1.0 + realized_R * POSITION_RISK_FRAC)
            in_pos = 0; entry_price=None; stop=None; risk_per_unit=None
            k += 1

        equity_pct[i] = equity_R[i]

    daily_ret = np.zeros(len(df), dtype=float)
    if len(df) > 1:
        daily_ret[1:] = np.diff(equity_pct) / equity_pct[:-1]

    return pd.DataFrame({"equity_pct":equity_pct, "daily_ret":daily_ret, "equity_R":equity_R}, index=dates)

def R_metrics(trades: pd.DataFrame) -> Dict[str, float]:
    if trades is None or len(trades) == 0:
        return {"trades":0,"win_rate":np.nan,"profit_factor":np.nan,
                "total_R":0.0,"avg_R":0.0,"maxDD_R":0.0,"MAR_R":np.nan}
    wins = trades.loc[trades["realized_R"]>0,"realized_R"].sum()
    losses = abs(trades.loc[trades["realized_R"]<0,"realized_R"].sum())
    total_R = trades["realized_R"].sum()
    pf = (wins / losses) if losses > 0 else np.nan
    maxDD_R = 0.0
    cum = 0.0; peak = 0.0
    for r in trades["realized_R"].values:
        cum += r
        peak = max(peak, cum)
        dd = cum - peak
        if dd < maxDD_R:
            maxDD_R = dd
    MAR_R = (total_R / abs(maxDD_R)) if maxDD_R < 0 else np.nan
    return {
        "trades": int(len(trades)),
        "win_rate": float((trades["realized_R"]>0).mean()),
        "profit_factor": float(pf) if not np.isnan(pf) else np.nan,
        "total_R": float(total_R),
        "avg_R": float(trades["realized_R"].mean()),
        "maxDD_R": float(maxDD_R),
        "MAR_R": float(MAR_R) if not np.isnan(MAR_R) else np.nan
    }

# --------------------------
# Baseline signal generators
# --------------------------
def baseline_flat(df: pd.DataFrame) -> pd.Series:
    return pd.Series(0, index=df.index)

def baseline_random(df: pd.DataFrame, p_enter: float = 0.03, seed: int = 123) -> pd.Series:
    rng = np.random.default_rng(seed)
    sig = np.zeros(len(df), dtype=int)
    in_pos = 0
    for i in range(len(df)-1):
        if in_pos == 0 and rng.uniform() < p_enter:
            sig[i] = 1 if rng.uniform() < 0.5 else 2
            in_pos = sig[i]
        elif in_pos != 0:
            sig[i] = 0  # no manual close
    return pd.Series(sig, index=df.index)

def baseline_trend(df: pd.DataFrame, fast: int = 50, slow: int = 200) -> pd.Series:
    ma_fast = df["close"].ewm(span=fast, adjust=False).mean()
    ma_slow = df["close"].ewm(span=slow, adjust=False).mean()
    above = (ma_fast > ma_slow).astype(int)
    cross = above.diff().fillna(0)
    sig = np.zeros(len(df), dtype=int)
    in_pos = 0
    for i in range(len(df)-1):
        if in_pos == 0:
            if cross.iloc[i] == +1:
                sig[i] = 1; in_pos = 1
            elif cross.iloc[i] == -1:
                sig[i] = 2; in_pos = 2
        else:
            sig[i] = 0  # no manual close
    return pd.Series(sig, index=df.index)

# --------------------------
# Plotting helpers (price + trades, equity-in-R)
# --------------------------
def plot_price_with_trades(df: pd.DataFrame, trades: pd.DataFrame, path_png: str, title: str):
    fig = plt.figure()
    plt.plot(df.index, df["close"].values, label="Close")
    for _, tr in trades.iterrows():
        ei = tr["entry_t"]; xi = tr["exit_t"]
        ep = tr["entry_price"]; xp = tr["exit_price"]
        tp = tr["take"]; sp = tr["stop"]
        if tr["type"] == "long":
            plt.scatter([ei], [ep], marker="^", s=60)
        else:
            plt.scatter([ei], [ep], marker="v", s=60)
        mark = {"tp":"s", "sl":"x", "time":"o", "eod":"P"}.get(tr["reason"], "o")
        plt.scatter([xi], [xp], marker=mark, s=60)
        plt.hlines([sp, tp], xmin=ei, xmax=xi, linestyles="dashed", linewidth=1)
    plt.title(title)
    plt.xlabel("Date"); plt.ylabel("Price")
    plt.tight_layout()
    plt.savefig(path_png, dpi=120)
    plt.close(fig)

def plot_equity_R(equity_R: np.ndarray, path_png: str, stats_text: str, title: str):
    fig = plt.figure()
    x = np.arange(len(equity_R))
    plt.plot(x, equity_R, label="Equity (R)")
    peak = np.maximum.accumulate(equity_R)
    plt.fill_between(x, equity_R, peak, alpha=0.2, where=equity_R<peak)
    plt.title(title + "\n" + stats_text)
    plt.xlabel("Trade #"); plt.ylabel("Equity (R-units applied)")
    plt.tight_layout()
    plt.savefig(path_png, dpi=120)
    plt.close(fig)


In [ ]:
# @title Per-eval validation logging + R-based model selection

from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import os, pandas as pd, numpy as np

def _backtest_model_on_slice(model, df_slice: pd.DataFrame, scaler) -> dict:
    mkt_df = df_slice[price_cols].copy()
    feat_scaled = scaler.transform(df_slice[feature_cols].copy())
    base = ForexEnv(
        market_df=mkt_df, feat_df=feat_scaled,
        window_size=WINDOW_SIZE, position_frac=POSITION_RISK_FRAC,
        sl_atr_mult=SL_ATR_MULT, tp_atr_mult=TP_ATR_MULT, max_bars_in_trade=MAX_BARS_IN_TRADE,
        transaction_cost=TRANSACTION_COST, slippage=SLIPPAGE,
        invalid_action_penalty=INVALID_ACTION_PENALTY,
        dd_penalty=DD_PENALTY, turnover_penalty=TURNOVER_PENALTY,
        seed=RANDOM_SEED
    )
    env = DummyVecEnv([lambda: ForexEnvSB3(base)])
    obs = env.reset()
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        if done[0]:
            break
    trades = pd.DataFrame(base.trades)
    eq_R = equity_R_from_trades(trades)
    mtm = daily_mtm_from_trades(mkt_df, trades)
    rmet = R_metrics(trades)
    sharpe = _daily_sharpe(mtm["daily_ret"].values) if len(mtm)>0 else np.nan
    mdd_pct = _max_drawdown(mtm["equity_pct"].values) if len(mtm)>0 else np.nan
    return {"trades":trades, "equity_R":eq_R, "mtm":mtm, "r":rmet, "sharpe":sharpe, "maxdd_pct":mdd_pct}

class RMetricsEvalCallback(EvalCallback):
    def __init__(self, eval_env, eval_freq, save_dir, val_df, scaler, verbose=1):
        super().__init__(eval_env=eval_env, best_model_save_path=save_dir, log_path=save_dir,
                         eval_freq=eval_freq, n_eval_episodes=1, deterministic=True, render=False, verbose=verbose)
        self.val_df = val_df
        self.scaler = scaler
        self.save_dir = save_dir
        self.best_score = -np.inf
        self.best_model_path = None

        self.csv_path = os.path.join(save_dir, "val_metrics.csv")
        with open(self.csv_path, "w") as f:
            f.write("step,total_R,PF,trades,maxDD_R,MAR_R,sharpe,mdd_pct\n")

    def _on_step(self) -> bool:
        super_ret = super()._on_step()

        if self.eval_freq > 0 and (self.n_calls % self.eval_freq == 0):
            res = _backtest_model_on_slice(self.model, self.val_df, self.scaler)
            r = res["r"]
            with open(self.csv_path, "a") as f:
                f.write(f"{self.model.num_timesteps},{r['total_R']},{r['profit_factor']},{r['trades']},{r['maxDD_R']},{r['MAR_R']},{res['sharpe']},{res['maxdd_pct']}\n")

            trades = res["trades"]
            if len(trades) > 0:
                stats_txt = f"R={r['total_R']:.2f} | PF={r['profit_factor']:.2f} | MAR={r['MAR_R'] if r['MAR_R'] is not None else np.nan:.2f} | Trades={r['trades']}"
                png_price = os.path.join(self.save_dir, "val_latest_price_trades.png")
                plot_price_with_trades(self.val_df, trades, png_price, "Validation: Price + Trades")
                png_eq = os.path.join(self.save_dir, "val_latest_equity_R.png")
                plot_equity_R(res['equity_R'], png_eq, stats_txt, "Validation: Equity in R")

            pf_ok = (r["profit_factor"] is not np.nan) and (r["profit_factor"] >= 1.0)
            trades_ok = r["trades"] >= 6
            dd_ok = r["maxDD_R"] >= -2.0
            mar = r["MAR_R"] if r["MAR_R"] is not None else np.nan
            score = mar if not np.isnan(mar) else r["total_R"]

            if pf_ok and trades_ok and dd_ok and (not np.isnan(score)) and score > self.best_score:
                path = os.path.join(self.save_dir, f"best_model_step_{self.model.num_timesteps}.zip")
                self.model.save(path)
                self.best_model_path = path
                self.best_score = score
                if self.verbose:
                    print(f"[RSelect] New best at step {self.model.num_timesteps}: score={score:.3f} (saved {path})")

        return super_ret


In [ ]:
# @title Training override: R-based model selection, per-fold charts, baselines, WF summary

from stable_baselines3 import DQN
from sb3_contrib import QRDQN, RecurrentPPO
from torch import nn
import torch as th  # needed for optimizer_class=th.optim.AdamW
import numpy as np, os, pandas as pd

try:
    QRDQN_N_QUANTILES
except NameError:
    QRDQN_N_QUANTILES = 51

def make_model(algo, env):
    if algo.upper() == "DQN":
        policy_kwargs = dict(
            activation_fn=nn.ReLU,
            net_arch=[256, 256],
            optimizer_class=th.optim.AdamW,
            optimizer_kwargs=dict(weight_decay=1e-5)
        )
        return DQN(
            "MlpPolicy", env,
            learning_rate=2e-5,
            buffer_size=250_000,
            learning_starts=10_000,
            batch_size=1024,
            tau=0.01,
            gamma=0.99,
            train_freq=4,
            gradient_steps=4,
            exploration_fraction=0.40,
            exploration_final_eps=0.10,
            policy_kwargs=policy_kwargs,
            verbose=1, seed=RANDOM_SEED, device="auto"
        )

    if algo.upper() == "QRDQN":
        policy_kwargs = dict(
            activation_fn=nn.ReLU,
            net_arch=[256, 256],
            n_quantiles=QRDQN_N_QUANTILES
        )
        return QRDQN(
            "MlpPolicy", env,
            learning_rate=3e-5,
            buffer_size=250_000,
            learning_starts=10_000,
            batch_size=1024,
            tau=0.01,
            gamma=0.99,
            train_freq=4,
            gradient_steps=2,
            policy_kwargs=policy_kwargs,
            verbose=1, seed=RANDOM_SEED, device="auto"
        )

    if algo.upper() == "RPPO":
        policy_kwargs = dict(activation_fn=nn.ReLU, net_arch=dict(pi=[128,128], vf=[128,128]))
        return RecurrentPPO(
            "MlpLstmPolicy", env,
            n_steps=RPPO_N_STEPS,
            batch_size=768,
            n_epochs=5,
            ent_coef=0.005,
            learning_rate=3e-5,
            clip_range=0.2,
            policy_kwargs=policy_kwargs,
            verbose=1, seed=RANDOM_SEED, device="auto"
        )

    raise ValueError(f"Unknown algo: {algo}")

def train_one_fold(algo, df_all, train_start, train_end, val_start, val_end, total_timesteps=None, eval_freq=None, n_envs=None):
    if total_timesteps is None:
        total_timesteps = TOTAL_TIMESTEPS if 'TOTAL_TIMESTEPS' in globals() else 1_200_000
    if eval_freq is None:
        eval_freq = EVAL_FREQ if 'EVAL_FREQ' in globals() else 140_000
    n_envs = n_envs or N_ENVS

    train_df = df_all.loc[train_start:train_end].copy()
    val_df   = df_all.loc[val_start:val_end].copy()

    scaler = ZScoreScaler()
    scaler.fit(train_df[feature_cols])

    if n_envs > 1:
        fns = [_make_one_env(train_df, scaler, RANDOM_SEED + i) for i in range(n_envs)]
        train_env = SubprocVecEnv(fns)
    else:
        train_env = DummyVecEnv([_make_one_env(train_df, scaler, RANDOM_SEED)])

    eval_env = DummyVecEnv([_make_one_env(val_df, scaler, RANDOM_SEED + 999)])

    model = make_model(algo, train_env)

    run_id = f"{algo}_fold_{str(train_end)[:10]}_{np.random.randint(1e9)}"
    save_dir = os.path.join(LOG_DIR, run_id); os.makedirs(save_dir, exist_ok=True)

    rcb = RMetricsEvalCallback(eval_env, eval_freq=eval_freq, save_dir=save_dir, val_df=val_df, scaler=scaler, verbose=1)

    # Progress bar (UI only; has no effect on learning dynamics)
    try:
        from stable_baselines3.common.callbacks import ProgressBarCallback
        _pb = ProgressBarCallback() if ('USE_PROGRESS_BAR' in globals() and USE_PROGRESS_BAR) else None
    except Exception:
        _pb = None
    _callbacks = [rcb] + ([_pb] if _pb is not None else [])

    model.learn(total_timesteps=total_timesteps, callback=_callbacks)

    best_path = rcb.best_model_path
    if best_path and os.path.exists(best_path):
        if algo.upper() == "RPPO":
            model = RecurrentPPO.load(best_path, env=train_env, device="auto")
        elif algo.upper() == "QRDQN":
            model = QRDQN.load(best_path, env=train_env, device="auto")
        else:
            model = DQN.load(best_path, env=train_env, device="auto")

    try:
        train_env.close(); eval_env.close()
    except Exception:
        pass

    import gc
    del eval_env
    gc.collect()
    try:
        import torch as _th
        if _th.cuda.is_available():
            _th.cuda.empty_cache()
    except Exception:
        pass

    return model, save_dir, scaler

def run_backtest_for_agent(model, scaler, df_slice):
    res = _backtest_model_on_slice(model, df_slice, scaler)
    trades, eqR, mtm = res["trades"], res["equity_R"], res["mtm"]
    r = res["r"]
    sharpe, mdd_pct = res["sharpe"], res["maxdd_pct"]
    return trades, eqR, mtm, r, sharpe, mdd_pct

def run_baselines(df_slice):
    sig_flat = baseline_flat(df_slice)
    tr_flat = _simulate_trades_from_signals(df_slice, sig_flat, POSITION_RISK_FRAC, SL_ATR_MULT, TP_ATR_MULT, MAX_BARS_IN_TRADE)
    eqR_flat = equity_R_from_trades(tr_flat)
    mtm_flat = daily_mtm_from_trades(df_slice, tr_flat)
    r_flat = R_metrics(tr_flat)
    sharpe_flat = _daily_sharpe(mtm_flat["daily_ret"].values) if len(mtm_flat)>0 else np.nan
    mdd_flat = _max_drawdown(mtm_flat["equity_pct"].values) if len(mtm_flat)>0 else np.nan

    sig_rand = baseline_random(df_slice)
    tr_rand = _simulate_trades_from_signals(df_slice, sig_rand, POSITION_RISK_FRAC, SL_ATR_MULT, TP_ATR_MULT, MAX_BARS_IN_TRADE)
    eqR_rand = equity_R_from_trades(tr_rand)
    mtm_rand = daily_mtm_from_trades(df_slice, tr_rand)
    r_rand = R_metrics(tr_rand)
    sharpe_rand = _daily_sharpe(mtm_rand["daily_ret"].values) if len(mtm_rand)>0 else np.nan
    mdd_rand = _max_drawdown(mtm_rand["equity_pct"].values) if len(mtm_rand)>0 else np.nan

    sig_tr = baseline_trend(df_slice, 50, 200)
    tr_tr  = _simulate_trades_from_signals(df_slice, sig_tr, POSITION_RISK_FRAC, SL_ATR_MULT, TP_ATR_MULT, MAX_BARS_IN_TRADE)
    eqR_tr = equity_R_from_trades(tr_tr)
    mtm_tr = daily_mtm_from_trades(df_slice, tr_tr)
    r_tr = R_metrics(tr_tr)
    sharpe_tr = _daily_sharpe(mtm_tr["daily_ret"].values) if len(mtm_tr)>0 else np.nan
    mdd_tr = _max_drawdown(mtm_tr["equity_pct"].values) if len(mtm_tr)>0 else np.nan

    baselines = {
        "flat":   {"trades":tr_flat, "eqR":eqR_flat, "mtm":mtm_flat, "r":r_flat, "sharpe":sharpe_flat, "mdd_pct":mdd_flat},
        "random": {"trades":tr_rand, "eqR":eqR_rand, "mtm":mtm_rand, "r":r_rand, "sharpe":sharpe_rand, "mdd_pct":mdd_rand},
        "trend":  {"trades":tr_tr,   "eqR":eqR_tr,   "mtm":mtm_tr,   "r":r_tr,   "sharpe":sharpe_tr,  "mdd_pct":mdd_tr}
    }
    return baselines

def run_walkforward(algo, df_all, folds):
    results = []
    for f in folds:
        f_id = f["fold"]
        tr_s, tr_e = f["train_start"], f["train_end"]
        va_s, va_e = f["val_start"],   f["val_end"]
        te_s, te_e = f["test_start"],  f["test_end"]

        print(f"\n=== Fold {f_id}: Train {tr_s.date()}→{tr_e.date()} | Val {va_s.date()}→{va_e.date()} | Test {te_s.date()}→{te_e.date()} ===")

        model, save_dir, scaler = train_one_fold(algo, df_all, tr_s, tr_e, va_s, va_e, n_envs=WF_N_ENVS)

        # Validation
        val_df = df_all.loc[va_s:va_e].copy()
        v_trades, v_eqR, v_mtm, v_r, v_sharpe, v_mdd_pct = run_backtest_for_agent(model, scaler, val_df)
        if len(v_trades) > 0:
            plot_price_with_trades(val_df, v_trades, os.path.join(save_dir, f"fold{f_id}_VAL_price_trades.png"), f"Fold {f_id} VAL: Price + Trades")
            stats_txt = f"R={v_r['total_R']:.2f} | PF={v_r['profit_factor']:.2f} | MAR={v_r['MAR_R'] if v_r['MAR_R'] is not None else np.nan:.2f} | Trades={v_r['trades']}"
            plot_equity_R(v_eqR, os.path.join(save_dir, f"fold{f_id}_VAL_equity_R.png"), stats_txt, f"Fold {f_id} VAL: Equity in R")

        # Test
        test_df = df_all.loc[te_s:te_e].copy()
        t_trades, t_eqR, t_mtm, t_r, t_sharpe, t_mdd_pct = run_backtest_for_agent(model, scaler, test_df)
        if len(t_trades) > 0:
            plot_price_with_trades(test_df, t_trades, os.path.join(save_dir, f"fold{f_id}_TEST_price_trades.png"), f"Fold {f_id} TEST: Price + Trades")
            stats_txt = f"R={t_r['total_R']:.2f} | PF={t_r['profit_factor']:.2f} | MAR={t_r['MAR_R'] if t_r['MAR_R'] is not None else np.nan:.2f} | Trades={t_r['trades']}"
            plot_equity_R(t_eqR, os.path.join(save_dir, f"fold{f_id}_TEST_equity_R.png"), stats_txt, f"Fold {f_id} TEST: Equity in R")

        bas = run_baselines(test_df)

        exit_mix = t_trades["reason"].value_counts().to_dict() if len(t_trades)>0 else {}
        row = dict(
            fold=f_id,
            total_R=t_r["total_R"], avg_R=t_r["avg_R"], PF=t_r["profit_factor"], win_rate=t_r["win_rate"],
            maxDD_R=t_r["maxDD_R"], MAR_R=t_r["MAR_R"], trades=t_r["trades"],
            sharpe=t_sharpe, maxDD_pct=t_mdd_pct,
            return_pct=(t_mtm["equity_pct"].iloc[-1]-1.0) if len(t_mtm)>0 else 0.0,
            exit_mix=json.dumps(exit_mix),
            flat_R=bas["flat"]["r"]["total_R"], flat_PF=bas["flat"]["r"]["profit_factor"], flat_sharpe=bas["flat"]["sharpe"],
            rand_R=bas["random"]["r"]["total_R"], rand_PF=bas["random"]["r"]["profit_factor"], rand_sharpe=bas["random"]["sharpe"],
            trend_R=bas["trend"]["r"]["total_R"], trend_PF=bas["trend"]["r"]["profit_factor"], trend_sharpe=bas["trend"]["sharpe"],
        )
        results.append(row)

        del model, scaler, val_df, test_df, v_trades, t_trades, v_mtm, t_mtm, bas

    dfres = pd.DataFrame(results)
    if len(dfres) > 0:
        fig = plt.figure()
        x = np.arange(len(dfres))
        plt.bar(x-0.3, dfres["total_R"].values, width=0.2, label="Agent R")
        plt.bar(x-0.1, dfres["flat_R"].values, width=0.2, label="Flat R")
        plt.bar(x+0.1, dfres["rand_R"].values, width=0.2, label="Random R")
        plt.bar(x+0.3, dfres["trend_R"].values, width=0.2, label="Trend R")
        plt.xticks(x, [f"F{int(v)}" for v in dfres["fold"]])
        plt.title("Agent vs Baselines (total R)")
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, "wf_agent_vs_baselines_R.png"), dpi=130); plt.close(fig)

        fig = plt.figure()
        plt.boxplot(dfres["sharpe"].dropna().values, vert=True); plt.ylabel("Sharpe (daily MTM)")
        plt.title("Sharpe across folds"); plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, "wf_sharpe_box.png"), dpi=130); plt.close(fig)

        fig = plt.figure()
        plt.boxplot(dfres["maxDD_pct"].dropna().values, vert=True); plt.ylabel("Max DD (%)")
        plt.title("Max Drawdown across folds"); plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, "wf_maxdd_box.png"), dpi=130); plt.close(fig)

        fig = plt.figure()
        plt.scatter(dfres["trades"].values, dfres["PF"].values)
        plt.xlabel("# trades"); plt.ylabel("Profit Factor"); plt.title("Engagement vs Quality")
        plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, "wf_engagement_vs_quality.png"), dpi=130); plt.close(fig)

    return dfres


In [ ]:

# @title Trade Report + Extended Metrics (NEW)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, io, base64, json
from datetime import datetime

def _sortino(daily_returns: np.ndarray, rf: float = 0.0) -> float:
    if daily_returns is None or len(daily_returns) < 2:
        return np.nan
    downside = daily_returns.copy()
    downside[downside > rf/252.0] = 0.0
    dd = np.sqrt((downside**2).mean())
    if dd <= 1e-12:
        return np.nan
    mu = daily_returns.mean() - rf/252.0
    return float(mu / dd * np.sqrt(252))

def _equity_from_pct(daily_pct: np.ndarray, start=1.0) -> np.ndarray:
    e = [start]
    for r in daily_pct:
        e.append(e[-1] * (1.0 + r))
    return np.array(e)

def _drawdown_series(equity: np.ndarray) -> np.ndarray:
    peak = np.maximum.accumulate(equity)
    return (equity / peak) - 1.0

def _rolling_sharpe(daily_pct: np.ndarray, window: int = 63) -> np.ndarray:
    out = np.full_like(daily_pct, np.nan, dtype=float)
    for i in range(window, len(daily_pct)):
        w = daily_pct[i-window:i]
        mu = w.mean()
        sd = w.std(ddof=1)
        if sd > 1e-12:
            out[i] = (mu / sd) * np.sqrt(252)
    return out

def _monthly_returns(mtm_df: pd.DataFrame) -> pd.DataFrame:
    if mtm_df is None or len(mtm_df) == 0:
        return pd.DataFrame()
    df = mtm_df.copy()
    df = df.rename(columns={"daily_ret":"ret"})
    df["YYYY-MM"] = df.index.to_period("M").astype(str)
    return df.groupby("YYYY-MM")["ret"].sum().to_frame("ret")

def plot_underwater(daily_pct: np.ndarray, path_png: str, title: str):
    eq = _equity_from_pct(daily_pct, start=1.0)
    dd = _drawdown_series(eq)
    fig = plt.figure()
    x = np.arange(len(dd))
    plt.fill_between(x, dd, 0.0, alpha=0.3)
    plt.title(title)
    plt.xlabel("Day"); plt.ylabel("Drawdown")
    plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)

def plot_hist_R(trades: pd.DataFrame, path_png: str, title: str):
    fig = plt.figure()
    plt.hist(trades["realized_R"].values, bins=30)
    plt.title(title); plt.xlabel("R"); plt.ylabel("Count")
    plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)

def plot_rolling_sharpe(daily_pct: np.ndarray, path_png: str, title: str):
    rs = _rolling_sharpe(np.asarray(daily_pct, dtype=float), window=63)
    fig = plt.figure()
    plt.plot(rs)
    plt.title(title); plt.xlabel("Day"); plt.ylabel("Rolling Sharpe (63d)")
    plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)

def plot_monthly_table(mtm_df: pd.DataFrame, path_png: str, title: str):
    m = _monthly_returns(mtm_df)
    if len(m) == 0:
        # empty table placeholder
        fig = plt.figure()
        plt.title(title + " (no data)")
        plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)
        return

    # Build a small image table
    months = m.index.tolist()
    vals = m["ret"].values
    fig = plt.figure()
    plt.axis("off")
    cell_text = [[f"{v*100:.2f}%"] for v in vals]
    table = plt.table(cellText=cell_text, rowLabels=months, colLabels=["Monthly %"], loc="center")
    table.scale(1, 1.2)
    plt.title(title)
    plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)

def plot_price_with_trades_ts(df: pd.DataFrame, trades: pd.DataFrame, path_png: str, title: str):
    # safer: map integer bars in trades to the timestamp index
    fig = plt.figure()
    plt.plot(df.index, df["close"].values, label="Close")
    n = len(df)
    for _, tr in trades.iterrows():
        ei = int(tr["entry_t"]); xi = int(tr["exit_t"])
        if ei < 0 or xi >= n:
            continue
        ep = tr["entry_price"]; xp = tr["exit_price"]
        tp = tr.get("take", None); sp = tr.get("stop", None)
        et = df.index[ei]; xt = df.index[xi]
        if tr["type"] == "long":
            plt.scatter([et], [ep], marker="^", s=60)
        else:
            plt.scatter([et], [ep], marker="v", s=60)
        mark = {"tp":"s", "sl":"x", "time":"o", "eod":"P"}.get(tr.get("reason",""), "o")
        plt.scatter([xt], [xp], marker=mark, s=60)
        if sp is not None and tp is not None:
            plt.hlines([sp, tp], xmin=et, xmax=xt, linestyles="dashed", linewidth=1)
    plt.title(title); plt.xlabel("Date"); plt.ylabel("Price")
    plt.tight_layout(); plt.savefig(path_png, dpi=120); plt.close(fig)

def extended_metrics(trades: pd.DataFrame, mtm_df: pd.DataFrame) -> dict:
    # Base R metrics
    r = R_metrics(trades) if trades is not None else {}
    daily_pct = mtm_df["daily_ret"].values if (mtm_df is not None and "daily_ret" in mtm_df) else np.array([])
    sharpe = _daily_sharpe(daily_pct) if len(daily_pct) else np.nan  # from metrics cell
    sortino = _sortino(daily_pct) if len(daily_pct) else np.nan
    mdd_pct = _max_drawdown(mtm_df["equity_pct"].values) if (mtm_df is not None and "equity_pct" in mtm_df) else np.nan
    # Approx Calmar (annualized return over |MaxDD|) using daily returns
    if len(daily_pct):
        cagr = (1.0 + daily_pct).prod()**(252.0/len(daily_pct)) - 1.0
        calmar = (cagr / abs(mdd_pct)) if (mdd_pct is not None and not np.isnan(mdd_pct) and mdd_pct < 0) else np.nan
    else:
        cagr, calmar = np.nan, np.nan

    # Trade durations
    if trades is not None and len(trades) > 0:
        dur = (trades["exit_t"] - trades["entry_t"]).values
        avg_dur = float(np.mean(dur))
        med_dur = float(np.median(dur))
    else:
        avg_dur = med_dur = np.nan

    out = dict(
        trades=r.get("trades", 0), win_rate=r.get("win_rate", np.nan),
        profit_factor=r.get("profit_factor", np.nan), total_R=r.get("total_R", np.nan),
        avg_R=r.get("avg_R", np.nan), maxDD_R=r.get("maxDD_R", np.nan),
        MAR_R=r.get("MAR_R", np.nan), sharpe=sharpe, sortino=sortino,
        maxDD_pct=mdd_pct, CAGR=cagr, calmar=calmar,
        avg_bars_in_trade=avg_dur, med_bars_in_trade=med_dur
    )
    return out

def _to_base64(path_png: str) -> str:
    with open(path_png, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")

def generate_trade_report(df_slice: pd.DataFrame, trades: pd.DataFrame, mtm_df: pd.DataFrame, equity_R: np.ndarray, out_prefix: str = "/mnt/data/rl_report"):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    # Plots
    png_price = f"{out_prefix}_price_trades.png"
    png_equityR = f"{out_prefix}_equityR.png"
    png_under = f"{out_prefix}_underwater.png"
    png_histR  = f"{out_prefix}_histR.png"
    png_months = f"{out_prefix}_monthly.png"
    plot_price_with_trades_ts(df_slice, trades, png_price, "Price w/ Trades")
    stats_txt = ""
    plot_equity_R(equity_R, png_equityR, stats_txt, "Equity (R)")
    if mtm_df is not None and len(mtm_df) > 0:
        plot_underwater(mtm_df["daily_ret"].values, png_under, "Underwater (Drawdown)")
        plot_rolling_sharpe(mtm_df["daily_ret"].values, f"{out_prefix}_rolling_sharpe.png", "Rolling Sharpe (63d)")
        plot_monthly_table(mtm_df, png_months, "Monthly Returns")
    else:
        # create empty placeholders to avoid broken links
        for p in [png_under, f"{out_prefix}_rolling_sharpe.png", png_months]:
            fig = plt.figure(); plt.title("No data"); plt.tight_layout(); plt.savefig(p, dpi=120); plt.close(fig)

    plot_hist_R(trades, png_histR, "Distribution of Trade R")

    # Metrics
    met = extended_metrics(trades, mtm_df)
    with open(f"{out_prefix}_metrics.json", "w") as f:
        json.dump(met, f, indent=2)

    # CSV Trades
    trades.to_csv(f"{out_prefix}_trades.csv", index=False)

    # Simple HTML report (lightweight, inspired by MT report)
    html = f"""
<html>
<head><meta charset="utf-8"><title>RL Strategy Report</title></head>
<body style="font-family: Arial, sans-serif;">
<h2>RL Strategy Report</h2>
<p><b>Generated:</b> {datetime.utcnow().isoformat()}Z</p>

<h3>Summary</h3>
<table border="1" cellpadding="6" cellspacing="0">
<tr><td>Trades</td><td>{met['trades']}</td></tr>
<tr><td>Win Rate</td><td>{met['win_rate']:.2%}</td></tr>
<tr><td>Profit Factor</td><td>{met['profit_factor']:.3f}</td></tr>
<tr><td>Total R</td><td>{met['total_R']:.3f}</td></tr>
<tr><td>Avg R / Trade</td><td>{met['avg_R']:.3f}</td></tr>
<tr><td>Max DD (R)</td><td>{met['maxDD_R']:.3f}</td></tr>
<tr><td>MAR (R)</td><td>{met['MAR_R']:.3f}</td></tr>
<tr><td>Sharpe (daily)</td><td>{met['sharpe']:.3f}</td></tr>
<tr><td>Sortino (daily)</td><td>{met['sortino']:.3f}</td></tr>
<tr><td>Max DD (%)</td><td>{met['maxDD_pct']:.2%}</td></tr>
<tr><td>CAGR (daily)</td><td>{met['CAGR']:.2%}</td></tr>
<tr><td>Calmar</td><td>{met['calmar']:.3f}</td></tr>
<tr><td>Avg Bars in Trade</td><td>{met['avg_bars_in_trade']:.2f}</td></tr>
<tr><td>Median Bars in Trade</td><td>{met['med_bars_in_trade']:.2f}</td></tr>
</table>

<h3>Charts</h3>
<p><img src="rl_report_price_trades.png" width="800"></p>
<p><img src="rl_report_equityR.png" width="800"></p>
<p><img src="rl_report_underwater.png" width="800"></p>
<p><img src="rl_report_histR.png" width="600"></p>
<p><img src="rl_report_rolling_sharpe.png" width="800"></p>
<p><img src="rl_report_monthly.png" width="400"></p>

<h3>Trades (CSV)</h3>
<p>Saved to <code>{out_prefix}_trades.csv</code></p>

<h3>Raw Metrics (JSON)</h3>
<p>Saved to <code>{out_prefix}_metrics.json</code></p>
</body></html>
"""
    html_path = f"{out_prefix}.html"
    with open(html_path, "w") as f:
        f.write(html)
    return {
        "html": html_path,
        "images": [png_price, png_equityR, png_under, png_histR, f"{out_prefix}_rolling_sharpe.png", png_months],
        "csv": f"{out_prefix}_trades.csv",
        "json": f"{out_prefix}_metrics.json"
    }


In [ ]:

# @title Smoke Test Walk-Forward (quick sanity check)

# This cell trains **only the last fold** with a smaller number of timesteps
# to validate the full pipeline (training → backtest → metrics → report)
# without changing your main Config or runner. Toggle SMOKE_TEST=True to run.

SMOKE_TEST = True            # 👈 set to True and run this cell to execute
SMOKE_TIMESTEPS = 400_000      # smaller than TOTAL_TIMESTEPS

if SMOKE_TEST:
    last = folds[-1]
    # temporarily train with smaller budget
    _old = TOTAL_TIMESTEPS
    try:
        TOTAL_TIMESTEPS = SMOKE_TIMESTEPS
        model, save_dir, scaler = train_one_fold(ALGO, data, last["train_start"], last["train_end"], last["val_start"], last["val_end"], n_envs=WF_N_ENVS)
    finally:
        TOTAL_TIMESTEPS = _old

    # Backtest on test slice
    df_test = data.loc[last["test_start"]:last["test_end"]].copy()
    res = _backtest_model_on_slice(model, df_test, scaler)
    trades, eqR, mtm = res["trades"], res["equity_R"], res["mtm"]

    # Generate a compact HTML report
    out = generate_trade_report(df_test, trades, mtm, eqR, out_prefix="/mnt/data/rl_report_smoke")
    print("Smoke test artifacts:")
    print(out)


In [ ]:
# @title Full Walk-Forward CV (updated runner)
results_wf = run_walkforward(ALGO, data, folds)
display(results_wf)


In [ ]:

# @title Walk-Forward HTML Report & ZIP (runs after Full WF)
import os, json, zipfile
from datetime import datetime
import numpy as np

assert 'results_wf' in globals(), "Run the Full Walk-Forward CV cell first to create 'results_wf'."

# Ensure report dir exists
REPORT_DIR = os.path.join(LOG_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

# Save WF results to CSV (if not already)
csv_path = os.path.join(REPORT_DIR, "wf_results.csv")
try:
    results_wf.to_csv(csv_path, index=False)
except Exception:
    pass

# Aggregate metrics
def _agg(df):
    cols = ["total_R","PF","win_rate","trades","sharpe","maxDD_pct","MAR_R","return_pct"]
    out = {"mean":{}, "median":{}, "std":{}, "min":{}, "max":{}}
    for c in cols:
        if c in df.columns:
            x = df[c].astype(float).values
            if len(x):
                out["mean"][c] = float(np.nanmean(x))
                out["median"][c] = float(np.nanmedian(x))
                out["std"][c] = float(np.nanstd(x, ddof=1)) if len(x)>1 else np.nan
                out["min"][c] = float(np.nanmin(x))
                out["max"][c] = float(np.nanmax(x))
    return out

agg = _agg(results_wf)
with open(os.path.join(REPORT_DIR, "wf_aggregates.json"), "w") as f:
    json.dump(agg, f, indent=2)

# Build HTML
html_path = os.path.join(REPORT_DIR, "wf_report.html")
now = datetime.now().strftime("%Y-%m-%d %H:%M")
imgs = []
for fn in ["wf_agent_vs_baselines_R.png", "wf_sharpe_box.png", "wf_maxdd_box.png", "wf_engagement_vs_quality.png"]:
    p = os.path.join(LOG_DIR, fn)
    if os.path.exists(p): imgs.append(p)

with open(html_path, "w") as f:
    f.write(f"<h2>Walk-Forward Report — {ALGO}</h2>")
    f.write(f"<p>Generated: {now}</p>")
    f.write("<h3>Per-Fold Summary</h3>")
    f.write(results_wf.to_html(index=False))
    f.write("<h3>Aggregates</h3>")
    f.write("<pre>"+json.dumps(agg, indent=2)+"</pre>")
    for p in imgs:
        f.write(f"<h3>{os.path.basename(p)}</h3><img src='{p}' style='max-width:100%;height:auto;'>")

print("Saved HTML:", html_path)

# Also place Smoke report (if exists) into REPORT_DIR for easy zipping
smoke_html = "/mnt/data/rl_report_smoke.html"
if os.path.exists(smoke_html):
    import shutil
    shutil.copy2(smoke_html, os.path.join(REPORT_DIR, os.path.basename(smoke_html)))

# Zip packs
def _zip_dir(root_dir, out_zip):
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for base, _dirs, files in os.walk(root_dir):
            for fn in files:
                fp = os.path.join(base, fn)
                arc = os.path.relpath(fp, root_dir)
                z.write(fp, arcname=arc)
    return out_zip

# zip_reports = _zip_dir(REPORT_DIR, os.path.join(LOG_DIR, "reports_pack.zip"))
# zip_logs    = _zip_dir(LOG_DIR,     os.path.join(LOG_DIR, "logs_pack.zip"))
# print("Zipped reports:", zip_reports)
# print("Zipped logs:", zip_logs)

# html_path, zip_reports, zip_logs


### Save results in drive

In [ ]:
# Mount Drive, create "results", and copy /content/logs_rl -> /content/drive/MyDrive/results/logs_rl
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

SRC = "/content/logs_rl"
DST_ROOT = "/content/drive/MyDrive/results"
DST = os.path.join(DST_ROOT, "logs_rl")  # keeps the folder name

# Make sure source exists
if not os.path.exists(SRC):
    raise FileNotFoundError(f"Source folder not found: {SRC}")

# Create results folder (if missing)
os.makedirs(DST_ROOT, exist_ok=True)

# Copy everything (merges if it already exists, overwriting same-named files)
shutil.copytree(SRC, DST, dirs_exist_ok=True)

# Small summary
files = sum(len(f) for _, _, f in os.walk(DST))
dirs = sum(len(d) for _, d, _ in os.walk(DST))
print(f"Copied to: {DST}\nSummary: {dirs} folders, {files} files")
